In [1]:
import pandas as pd

## **Data Sources**



Six datasets were utilized, with three playing a direct role in constructing the customer master table:




| Dataset Name                 | Description                                          | Role                                |
| ---------------------------- | ---------------------------------------------------- | ----------------------------------- |
| churn.csv                    | Customer financial & demographic baseline            | **Primary baseline**                |
| bank_additional_full.csv     | Behavioral + marketing interaction data              | **Behavioral enrichment**           |
| marketing_campaign.csv (TSV) | Income, education, marital status, lifecycle info    | **Demographic & income enrichment** |

note: for the customer master data the three datasets will be unified with the churn dataset as the baseline.


In [2]:
csv = {
    "bank": r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\bank-additional-full.csv",   # ; separated
    "churn": r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\churn.csv",
    "marketing": r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\marketing_campaign.csv",
    }

In [3]:
def load_csvs():
    data = {}   # important!
    for key, path in csv.items():

        # detect delimiter based on file name
        if "bank-additional" in path:
            sep = ';'
        elif "marketing" in path:   # <-- your marketing_campaign.tsv
            sep = '\t'
        else:
            sep = ','

        print(f"Loading {path} using sep='{sep}' ...")
        df = pd.read_csv(path, sep=sep)

        print(f"  → Loaded {len(df):,} rows, {len(df.columns)} columns")
        data[key] = df

    return data


In [4]:
data = load_csvs()

Loading D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\bank-additional-full.csv using sep=';' ...
  → Loaded 41,188 rows, 21 columns
Loading D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\churn.csv using sep=',' ...
  → Loaded 10,000 rows, 14 columns
Loading D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\data raw\marketing_campaign.csv using sep='	' ...
  → Loaded 2,240 rows, 29 columns


## **CUSTOMER_MASTER_FULL – ETL Data Dictionary**

| Field Name        | Source Dataset                                          | Source Column | Transformation Logic                                                                  | Type         | Purpose                                   |
| ----------------- | ------------------------------------------------------- | ------------- | ------------------------------------------------------------------------------------- | ------------ | ----------------------------------------- |
| **customer_id**   | `churn.csv` → replaced by code                          | – (generated) | Generated using custom function `generate_customer_id()`                              | Synthetic ID | Unique customer identifier                |
| **first_name**    | Synthetic (Faker)                                       | –             | Faker-generated first name                                                            | Synthetic    | Identity enrichment                       |
| **surname**       | `churn.csv`                                             | Surname       | Direct mapping; lowercased during normalization                                       | Direct       | Identity field                            |
| **gender**        | `churn.csv`                                             | Gender        | Direct mapping → lowercased                                                           | Normalize    | Demographic attribute                     |
| **date_of_birth** | Derived                                                 | –             | `current_year - age` using: `date_of_birth = current_year - Age`                      | Derived      | Generates DOB from age                    |
| **age**           | `churn.csv`                                             | Age           | Direct mapping → lowercased column name                                               | Statistical  | Key demographic indicator                 |
| **marital**       | `bank_additional_full.csv`                              | marital       | Direct mapping → lowercased                                                           | Direct       | Customer marital group                    |
| **education**     | `marketing_campaign.csv`                                | Education     | Standardized mappings (“basic”, “graduate”, etc.) then lowercased                     | Standardized | Categorical for modeling                  |
| **job**           | `bank_additional_full.csv`                              | job           | Direct mapping → title-case standardization → lowercased                              | Standardized | Socio-economic indicator                  |
| **income**        | `marketing_campaign.csv`                                | Income        | Direct mapping                                                                        | Statistical  | Income feature                            |
| **contact**       | `bank_additional_full.csv`                              | contact       | Standardize contact method → lowercased                                               | Standardized | Marketing segmentation                    |
| **nationality**   | `churn.csv` (original: Geography)                       | Geography     | Renamed to `nationality` (cell 40) → lowercased                                       | Normalize    | Country-of-origin marker                  |
| **balance**       | `churn.csv`                                             | Balance       | Direct mapping                                                                        | Statistical  | Financial position                        |
| **has_cr_card**   | `churn.csv`                                             | HasCrCard     | Renamed to `has_cr_card`, integer preserved                                           | Direct       | Product usage                             |
| **default**       | `bank_additional_full.csv`                              | default       | Direct mapping → lowercased                                                           | Normalize    | Credit behavior                           |
| **housing**       | `bank_additional_full.csv`                              | housing       | Direct mapping → lowercased                                                           | Normalize    | Housing loan indicator                    |
| **loan**          | `bank_additional_full.csv`                              | loan          | Direct mapping → lowercased                                                           | Normalize    | Personal loan indicator                   |
| **credit_score**  | `churn.csv`                                             | CreditScore   | Renamed to `credit_score`, numeric                                                    | Statistical  | Creditworthiness metric                   |
| **tenure**        | `churn.csv`                                             | Tenure        | Direct mapping                                                                        | Statistical  | Customer longevity                        |
| **dt_customer**   | `marketing_campaign.csv`                                | Dt_Customer   | Renamed to `dt_customer`; parsed as date                                              | Direct       | Customer join date                        |
| **created_date**  | `marketing_campaign.csv` (in code you copy dt_customer) | Dt_Customer   | In your code: `created_date = dt_customer` (duplicate of dt_customer, no other logic) | Derived      | Timestamp placeholder for future auditing |


## **ETL PIPELINE OVERVIEW**

The ETL pipeline consists of six major steps:

1. Extraction
Customer data is loaded from three source files: churn.csv, bank_additional_full.csv, and marketing_campaign.csv.

2. Standardization & Normalization
Column names are converted to lowercase, selected fields are renamed for consistency (e.g., CustomerId → customer_id, Geography → nationality), and categorical fields are normalized.

3. Resampling to Align Dataset Sizes
Source datasets with different row counts are resampled using random sampling with replacement to ensure consistent lengths before merging.

4. Merging into a Unified Schema
The standardized datasets are combined horizontally using pd.concat(...) to create a unified customer dataset.

5. Synthetic Identity Enrichment (Faker)
Synthetic first_name values are generated; surname is filled using real data with Faker as fallback. This provides privacy-preserving identity fields.

6. Feature Derivation (DOB)
A date_of_birth field is derived using the formula current_year - age.

7. Export to CUSTOMER_MASTER_FULL.csv
The final set of standardized customer attributes is exported as CUSTOMER_MASTER_FULL.csv.

8. Export to SQL Server (if needed)
The same dataset can optionally be written into a SQL table for downstream GRC or audit processes.


**A Data lineage is illustrated below:**

## **DATA LINEAGE DIAGRAM**

```mermaid
flowchart LR
    A["churn.csv\n(Baseline Financial & Demographic)"] --> B1[Standardize & Rename]
    B["bank_additional_full.csv\n(Behavioral & Socio-economic)"] --> B2[Standardize & Rename]
    D["marketing_campaign.csv\n(Income, Education, dt_customer)"] --> B3[Standardize & Rename]

    B2 --> R[Resample to Match Row Count]
    B3 --> R
    B1 --> R

    R --> F[Merge into Unified Customer Dataset]

    G["Faker Enrichment\n(first_name, fallback surname)"] --> F
    DOB["Feature Derivation\n(date_of_birth from age)"] --> F

    F --> H["CUSTOMER_MASTER_FULL.csv\n(Final Unified Dataset)"]
```

## **ERD DIAGRAM**

```mermaid
erDiagram

    CUSTOMER_MASTER_FULL {
        string customer_id
        string first_name
        string surname
        string gender
        date date_of_birth
        int age
        string marital
        string education
        string job
        float income
        string contact
        string nationality
        float balance
        int has_cr_card
        string default
        string housing
        string loan
        int credit_score
        int tenure
        date dt_customer
    }

    CHURN {
        string customerid
        string surname
        int creditscore
        string geography
        string gender
        int age
        int tenure
        float balance
        int hascrcard
    }

    BANK_ADDITIONAL {
        string job
        string default
        string housing
        string loan
        string contact
        string marital
    }

    MARKETING {
        int id
        string education
        float income
        date dt_customer
    }

    CHURN ||--o{ CUSTOMER_MASTER_FULL : provides
    BANK_ADDITIONAL ||--o{ CUSTOMER_MASTER_FULL : enriches
    MARKETING ||--o{ CUSTOMER_MASTER_FULL : enriches



```


## **SQL SCHEMA — CUSTOMER_MASTER_FULL**

```
CREATE TABLE CUSTOMER_MASTER_FULL (
    customer_id      VARCHAR(50)    NOT NULL PRIMARY KEY,
    first_name       VARCHAR(100)   NULL,
    surname          VARCHAR(100)   NULL,
    gender           VARCHAR(10)    NULL,
    date_of_birth    DATE           NULL,
    age              INT            NULL,
    marital          VARCHAR(20)    NULL,
    education        VARCHAR(50)    NULL,
    job              VARCHAR(100)   NULL,
    income           DECIMAL(18,2)  NULL,
    contact          VARCHAR(50)    NULL,
    nationality      VARCHAR(50)    NULL,
    balance          DECIMAL(18,2)  NULL,
    has_cr_card      TINYINT        NULL,
    [default]        VARCHAR(10)    NULL,  -- reserved word, so wrapped in []
    housing          VARCHAR(10)    NULL,
    loan             VARCHAR(10)    NULL,
    credit_score     INT            NULL,
    tenure           INT            NULL,
    dt_customer      DATE           NULL
);

```

### **Extract Phase**

In this phase, all raw datasets are imported into the ETL workflow. The primary baseline dataset is churn.csv (10,000 rows), chosen because it contains the core demographic and financial attributes needed for statistical modelling. Other datasets include:

- bank_additional_full.csv → behavioral and socio-economic attributes (job, loan, housing, contact, marital)

- marketing_campaign.csv → income, education (cleaned), and customer enrollment date (dt_customer)

- Faker synthetic generator → identity fields (first_name and fallback surname when missing)

Each file is loaded with the appropriate delimiter and basic type parsing before entering the transformation steps.

### **Churn Data as a Baseline**


Due to the different data dimensions across sources, churn.csv is used as the baseline because its completeness is better suited for creating the unified customer dataset. Consequently, the other datasets are resampled to fit the 10,000-record size of the churn dataset. This is done to ensure that the final merged data will be:

- approximately preserving the original statistical distributions of each source

- avoiding simple, deterministic cycling of records by using random sampling with replacement

- producing a balanced final dataset where each customer record has demographic, behavioral, and income-related attributes


In [5]:
df_churn = data["churn"]
N = len(df_churn)  # Baseline size

print(N)

10000


In [6]:
df_churn.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## **Standardization & Normalization**

The bank dataset contains noisy categorical fields, requiring transformation:

- Education (bank): mapped into BASIC, HIGH_SCHOOL, UNIVERSITY, etc.

- Education (marketing): mapped into UNIVERSITY, PHD, MASTER, BASIC

- Marital status: consolidated into SINGLE, MARRIED, DIVORCED, WIDOWED

- Contact: standardized to MOBILE, LANDLINE, UNKNOWN

- default/housing/loan: normalized YES/NO/UNKNOWN

This step ensures categorical consistency required for the next process.

### **Standardize & Normalize Data**

In [7]:
# call bank data as bank-additional-full.csv

df_bank = data["bank"]

# call marketing data as marketing_campaign.csv
df_marketing = data["marketing"]

In [8]:
df_bank.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [9]:
df_marketing.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [10]:
df_marketing['Education'].unique()
df_marketing['Marital_Status'].unique()

array(['Single', 'Together', 'Married', 'Divorced', 'Widow', 'Alone',
       'Absurd', 'YOLO'], dtype=object)

In [11]:
# Mapping from bank data

# Marital Status Standardization
df_bank['marital'] = df_bank['marital'].str.upper().replace({
    'MARRIED': 'MARRIED',
    'SINGLE': 'SINGLE',
    'DIVORCED': 'DIVORCED',
    'UNKNOWN': 'UNKNOWN'
})

# Contact Standardization
df_bank['contact'] = df_bank['contact'].replace({
    'cellular': 'MOBILE',
    'telephone': 'LANDLINE',
    'unknown': 'UNKNOWN'
})

# Default / housing / loan normalization
for col in ['default', 'housing', 'loan']:
    df_bank[col] = df_bank[col].replace({
        'yes': 'YES',
        'no': 'NO',
        'unknown': 'UNKNOWN'
    }).str.upper()


In [12]:
df_bank.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,MARRIED,basic.4y,NO,NO,NO,LANDLINE,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,MARRIED,high.school,UNKNOWN,NO,NO,LANDLINE,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,MARRIED,high.school,NO,YES,NO,LANDLINE,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,MARRIED,basic.6y,NO,NO,NO,LANDLINE,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,MARRIED,high.school,NO,NO,YES,LANDLINE,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [13]:
df_bank.shape

(41188, 21)

In [14]:
# Clean & normalize marketing_campaign (demographic + income)

df_marketing_clean = df_marketing[[
    'Income',
    'Dt_Customer',
    'Education',
]].copy()

# Standardize Education (marketing version)
edu_map_marketing_clean = {
    'Graduation': 'UNIVERSITY',
    'PhD': 'PHD',
    'Master': 'MASTER',
    '2n Cycle': 'PROFESSIONAL',
    'Basic': 'BASIC'
}
df_marketing_clean ['Education'] = df_marketing_clean ['Education'].replace(edu_map_marketing_clean).fillna('UNKNOWN')


# Parse Dt_Customer to proper date
df_marketing_clean['Dt_Customer'] = pd.to_datetime(
    df_marketing_clean['Dt_Customer'], 
    dayfirst=True, # to handle different date format
    errors='coerce'
)


## **Resize Data Marketing & Bank adjusting with Data Churn**

In [15]:
# Resample bank to size N
df_bank_resized = df_bank.sample(N, replace=True).reset_index(drop=True)

# Resample marketing to size N
df_marketing_clean_resized = df_marketing_clean.sample(N, replace=True).reset_index(drop=True)


In [16]:
df_marketing_clean_resized.head()

,Income,Dt_Customer,Education
0,38620.0,2013-05-11,MASTER
1,87195.0,2014-05-08,UNIVERSITY
2,50014.0,2014-01-22,MASTER
3,10245.0,2013-05-15,UNIVERSITY
4,20895.0,2012-10-06,UNIVERSITY


In [17]:
df_marketing_clean_resized.shape

(10000, 3)

In [18]:
df_bank_resized.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,26,services,SINGLE,high.school,NO,NO,NO,LANDLINE,jun,fri,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.967,5228.1,no
1,32,management,MARRIED,university.degree,NO,YES,NO,MOBILE,jun,mon,...,3,999,0,nonexistent,-2.9,92.963,-40.8,1.260,5076.2,no
2,33,blue-collar,MARRIED,basic.4y,NO,NO,NO,LANDLINE,may,fri,...,2,999,0,nonexistent,1.1,93.994,-36.4,4.864,5191.0,no
3,56,retired,MARRIED,basic.6y,NO,YES,NO,LANDLINE,apr,mon,...,1,999,0,nonexistent,-1.8,93.075,-47.1,1.466,5099.1,no
4,52,technician,MARRIED,high.school,NO,NO,NO,LANDLINE,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no


In [19]:
df_bank_resized.shape

(10000, 21)

## **Merge All Data into new table 'CUSTOMER_MASTER_FULL'**

Now we horizontally concatenate with this alignment:

- original churn columns for the demographics data

- selected behavioral columns from bank

- selected columns from marketing to show income and lifecycle of the data

In [20]:
df_churn_reset = df_churn.reset_index(drop=True)

df_final = pd.concat([
    df_churn_reset[[
        "CustomerId", "Surname", "CreditScore", "Geography",
        "Gender", "Age", "Tenure", "Balance", "HasCrCard"
    ]],
    df_bank_resized[["job", "default", "housing", "loan", "contact", "marital"]],
    df_marketing_clean_resized[["Education", "Income", "Dt_Customer"]]
], axis=1)


In [21]:
df_final.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,HasCrCard,job,default,housing,loan,contact,marital,Education,Income,Dt_Customer
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,services,NO,NO,NO,LANDLINE,SINGLE,MASTER,38620.0,2013-05-11
1,15647311,Hill,608,Spain,Female,41,1,83807.86,0,management,NO,YES,NO,MOBILE,MARRIED,UNIVERSITY,87195.0,2014-05-08
2,15619304,Onio,502,France,Female,42,8,159660.80,1,blue-collar,NO,NO,NO,LANDLINE,MARRIED,MASTER,50014.0,2014-01-22
3,15701354,Boni,699,France,Female,39,1,0.00,0,retired,NO,YES,NO,LANDLINE,MARRIED,UNIVERSITY,10245.0,2013-05-15
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,technician,NO,NO,NO,LANDLINE,MARRIED,UNIVERSITY,20895.0,2012-10-06


In [22]:
df_final.shape

(10000, 18)

In [23]:
#change column name for geography and all column to uppercase

# column name for geography to country
df_final = df_final.rename(columns={'Geography': 'nationality',
                                    'CustomerId':'customer_id',
                                    'HasCrCard':'has_cr_card','CreditScore':'credit_score',
                                    'Dt_Customer':'dt_customer',})

#column name standardize to uppercase
df_final.columns = df_final.columns.str.lower()


In [24]:
# check the new dataframe information

df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   customer_id   10000 non-null  int64         
 1   surname       10000 non-null  object        
 2   credit_score  10000 non-null  int64         
 3   nationality   10000 non-null  object        
 4   gender        10000 non-null  object        
 5   age           10000 non-null  int64         
 6   tenure        10000 non-null  int64         
 7   balance       10000 non-null  float64       
 8   has_cr_card   10000 non-null  int64         
 9   job           10000 non-null  object        
 10  default       10000 non-null  object        
 11  housing       10000 non-null  object        
 12  loan          10000 non-null  object        
 13  contact       10000 non-null  object        
 14  marital       10000 non-null  object        
 15  education     10000 non-null  object 

In [25]:
# check the new dataframe dimension

df_final.shape

(10000, 18)

In [26]:
df_final.head()

,customer_id,surname,credit_score,nationality,gender,age,tenure,balance,has_cr_card,job,default,housing,loan,contact,marital,education,income,dt_customer
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,services,NO,NO,NO,LANDLINE,SINGLE,MASTER,38620.0,2013-05-11
1,15647311,Hill,608,Spain,Female,41,1,83807.86,0,management,NO,YES,NO,MOBILE,MARRIED,UNIVERSITY,87195.0,2014-05-08
2,15619304,Onio,502,France,Female,42,8,159660.80,1,blue-collar,NO,NO,NO,LANDLINE,MARRIED,MASTER,50014.0,2014-01-22
3,15701354,Boni,699,France,Female,39,1,0.00,0,retired,NO,YES,NO,LANDLINE,MARRIED,UNIVERSITY,10245.0,2013-05-15
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,technician,NO,NO,NO,LANDLINE,MARRIED,UNIVERSITY,20895.0,2012-10-06


## **Add synthetic identity fields (Faker)**

Based on the data availability, several identity-related fields are enriched synthetically to make the dataset more realistic. These identity fields do not affect downstream statistical analysis. According to the data dictionary, the synthetic attributes include:

- first_name (always generated using Faker)

- surname (Faker fallback only when missing in the churn dataset)

- customer_id (synthetically generated using a custom pattern, replacing the original CustomerId)

**Purpose:**

- ensure privacy of identifiable information

- provide realistic CRM-style identity attributes

- maintain statistical independence from real-world individuals

In [27]:
from faker import Faker
import random
fake = Faker()

In [28]:
# Add synthetic firstname
df_final["first_name"] = [fake.first_name() for _ in range(N)]

# If any Surname missing, fill with Faker
df_final["surname"] = df_final["surname"].fillna(
    pd.Series([fake.last_name() for _ in range(N)])
)

# regenerate CustomerId to a purely synthetic one
#df_final["customer_id"] = [str(fake.uuid4()) for _ in range(N)]

def generate_customer_id():
    return f"C{random.randint(1000, 9999)}"

df_final["customer_id"] = [generate_customer_id() for _ in range(len(df_final))]


In [29]:
df_final.head()

,customer_id,surname,credit_score,nationality,gender,age,tenure,balance,has_cr_card,job,default,housing,loan,contact,marital,education,income,dt_customer,first_name
0,C1845,Hargrave,619,France,Female,42,2,0.00,1,services,NO,NO,NO,LANDLINE,SINGLE,MASTER,38620.0,2013-05-11,Joseph
1,C7129,Hill,608,Spain,Female,41,1,83807.86,0,management,NO,YES,NO,MOBILE,MARRIED,UNIVERSITY,87195.0,2014-05-08,Briana
2,C3732,Onio,502,France,Female,42,8,159660.80,1,blue-collar,NO,NO,NO,LANDLINE,MARRIED,MASTER,50014.0,2014-01-22,Christopher
3,C9122,Boni,699,France,Female,39,1,0.00,0,retired,NO,YES,NO,LANDLINE,MARRIED,UNIVERSITY,10245.0,2013-05-15,Patricia
4,C3230,Mitchell,850,Spain,Female,43,2,125510.82,1,technician,NO,NO,NO,LANDLINE,MARRIED,UNIVERSITY,20895.0,2012-10-06,Robert


## **Feature  Tenure**


In the original dataset, there is tenure information, that reflect the longevity of customer join, however to make it consistent with the customer data, I will update the information.

In [30]:
current = pd.Timestamp("2025-01-01")
df_final["tenure"] = current.year - df_final["dt_customer"].dt.year

In [31]:
df_final["dt_customer"] = pd.to_datetime(df_final["dt_customer"], errors="coerce")

## **Feature  Date of Birth (DOB)**


In the original dataset, there is no DOB information, but only age. so, I will change it to the DOB format by assuming the age are the real condition in 2025, and I will process it by derived from subtracting the assumed date of January 1, 2025 with age

In [32]:
def generate_dob_from_age(age):
    current_year = 2025
    year = current_year - age
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    return pd.Timestamp(year=year, month=month, day=day)


df_final["date_of_birth"] = df_final["age"].apply(generate_dob_from_age)


In [33]:
df_final.head()

,customer_id,surname,credit_score,nationality,gender,age,tenure,balance,has_cr_card,job,default,housing,loan,contact,marital,education,income,dt_customer,first_name,date_of_birth
0,C1845,Hargrave,619,France,Female,42,12,0.00,1,services,NO,NO,NO,LANDLINE,SINGLE,MASTER,38620.0,2013-05-11,Joseph,1983-09-19
1,C7129,Hill,608,Spain,Female,41,11,83807.86,0,management,NO,YES,NO,MOBILE,MARRIED,UNIVERSITY,87195.0,2014-05-08,Briana,1984-11-23
2,C3732,Onio,502,France,Female,42,11,159660.80,1,blue-collar,NO,NO,NO,LANDLINE,MARRIED,MASTER,50014.0,2014-01-22,Christopher,1983-02-18
3,C9122,Boni,699,France,Female,39,12,0.00,0,retired,NO,YES,NO,LANDLINE,MARRIED,UNIVERSITY,10245.0,2013-05-15,Patricia,1986-04-02
4,C3230,Mitchell,850,Spain,Female,43,13,125510.82,1,technician,NO,NO,NO,LANDLINE,MARRIED,UNIVERSITY,20895.0,2012-10-06,Robert,1982-03-23


In [34]:
df_final.columns

Index(['customer_id', 'surname', 'credit_score', 'nationality', 'gender',
       'age', 'tenure', 'balance', 'has_cr_card', 'job', 'default', 'housing',
       'loan', 'contact', 'marital', 'education', 'income', 'dt_customer',
       'first_name', 'date_of_birth'],
      dtype='object')

In [36]:
df_final = df_final[[
    'customer_id',
    'first_name',
    'surname', 
    'gender', 
    'date_of_birth', 
    'age','marital', 
    'education', 
    'job', 
    'income', 
    'contact', 
    'nationality', 
    'balance', 
    'has_cr_card', 
    'default', 
    'housing', 
    'loan',  
    'credit_score', 
    'tenure', 
    'dt_customer']]


In [37]:
df_final.shape

(10000, 20)

In [38]:
df_final.head()

,customer_id,first_name,surname,gender,date_of_birth,age,marital,education,job,income,contact,nationality,balance,has_cr_card,default,housing,loan,credit_score,tenure,dt_customer
0,C1845,Joseph,Hargrave,Female,1983-09-19,42,SINGLE,MASTER,services,38620.0,LANDLINE,France,0.00,1,NO,NO,NO,619,12,2013-05-11
1,C7129,Briana,Hill,Female,1984-11-23,41,MARRIED,UNIVERSITY,management,87195.0,MOBILE,Spain,83807.86,0,NO,YES,NO,608,11,2014-05-08
2,C3732,Christopher,Onio,Female,1983-02-18,42,MARRIED,MASTER,blue-collar,50014.0,LANDLINE,France,159660.80,1,NO,NO,NO,502,11,2014-01-22
3,C9122,Patricia,Boni,Female,1986-04-02,39,MARRIED,UNIVERSITY,retired,10245.0,LANDLINE,France,0.00,0,NO,YES,NO,699,12,2013-05-15
4,C3230,Robert,Mitchell,Female,1982-03-23,43,MARRIED,UNIVERSITY,technician,20895.0,LANDLINE,Spain,125510.82,1,NO,NO,NO,850,13,2012-10-06


In [39]:
print(df_final.duplicated().sum())

0


In [40]:
df_final = (
    df_final
    .drop_duplicates(subset="customer_id", keep="first")
    .reset_index(drop=True)
)

print("Unique customers:", df_final["customer_id"].nunique())

Unique customers: 6044


In [41]:
# Eksport final data to csv (folder final source)

## df_final.to_csv(
##    r"D:\KULIAH\SEMESTER 4\DRAFT\Step by step\Final Source Data\CUSTOMER_MASTER_FULL.csv",
##    index=False
##)

df_final.to_csv("TRANSACTION_MASTER_FULL.csv", index=False)